# AHỤIKE 03 — Benchmark AHỤIKE vs base N-ATLAS, then live demo

Kaggle settings: **GPU T4 x2**, **Internet on**, Secret **HF_TOKEN** attached.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/EgwuSamuel/ahuike.git ahuike || (cd ahuike && git pull -q)
%cd ahuike

Pre-flight check: token, gated access to N-ATLaS + 4 ASR models, GPUs, chat template. Everything should say PASS before you continue.

In [ ]:
!python scripts/check_setup.py

In [ ]:
!nvidia-smi | head -4
!pip install -q vllm
# vLLM installs its own PyTorch build. Kaggle's preinstalled torchaudio no longer matches it and
# breaks every transformers import, so remove it (AHỤIKE does not need torchaudio).
!pip uninstall -y -q torchaudio
!python -c "import torch, vllm; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPUs visible:', torch.cuda.device_count(), '| vllm', vllm.__version__)"
# GPUs visible must be 2. If it is 0, the GPU driver is too old for this PyTorch build:
# Run > Factory reset, re-run the setup cells, SKIP this cell, and use --backend hf below.

In [ ]:
!python scripts/hub_sync.py pull data results outputs/ahuike-lora outputs/ablation-lora

In [ ]:
!python scripts/run_eval.py --backend auto --systems ahuike=outputs/ahuike-lora ablation=outputs/ablation-lora

In [ ]:
!python scripts/compute_metrics.py
!python scripts/hub_sync.py push results

## Live demo (restart the session first to free GPU memory from vLLM, re-run setup cells)
Opens a public Gradio link: use it for the video and for live user sessions.

In [ ]:
!pip install -q gradio peft
!python scripts/hub_sync.py pull data outputs/ahuike-lora
!python app/app.py --backend hf --adapter outputs/ahuike-lora --share

## Voice evaluation (after uploading recordings to voice/ and filling voice/manifest.csv)

In [ ]:
!python scripts/eval_voice.py --backend hf --adapter outputs/ahuike-lora